In [ ]:
# ============================================================
# COMPLETE MEMORY / CACHE / VARIABLE CLEANUP
# ============================================================

import os, gc, time, psutil

# ============================================================
# USER VARIABLES
# ============================================================

def get_user_variables():
    try:
        from IPython import get_ipython
        ip = get_ipython()
        if ip is None: return {}
        excluded = {"In", "Out", "get_ipython", "exit", "quit"}
        return {name: value for name, value in ip.user_ns.items()
                if not name.startswith("_") and name not in excluded}
    except Exception:
        return {}

# ============================================================
# RAM USAGE
# ============================================================

def get_ram_usage():
    return psutil.Process(os.getpid()).memory_info().rss / 1024**2

# ============================================================
# VARIABLE INFORMATION
# ============================================================

def print_variable_information():
    variables = get_user_variables()
    print(f"Variable count  : {len(variables)}")
    if variables: print("Variables       : " + ", ".join(sorted(variables.keys())))

# ============================================================
# CLEAR MATPLOTLIB
# ============================================================

def clear_matplotlib_cache():
    try:
        import matplotlib.pyplot as plt
        print("\n[1] Closing Matplotlib figures...")
        plt.close("all")
        print("    Matplotlib figures closed.")
    except ImportError:
        print("\n[1] Matplotlib not installed.")
    except Exception as e:
        print(f"\n[1] Matplotlib warning: {e}")

# ============================================================
# CLEAR TENSORFLOW / KERAS
# ============================================================

def clear_tensorflow_memory():
    try:
        import tensorflow as tf
        print("\n[2] Clearing TensorFlow / Keras...")
        tf.keras.backend.clear_session()
        print("    Keras session cleared.")

        gpus = tf.config.list_physical_devices("GPU")

        if gpus:
            for i in range(len(gpus)):
                try:
                    info = tf.config.experimental.get_memory_info(f"GPU:{i}")
                    print(f"    GPU:{i} current: {info['current']/1024**2:.2f} MB")
                    print(f"    GPU:{i} peak   : {info['peak']/1024**2:.2f} MB")
                except Exception:
                    pass

                try: tf.config.experimental.reset_memory_stats(f"GPU:{i}")
                except Exception: pass

        print("    TensorFlow cleanup completed.")

    except ImportError:
        print("\n[2] TensorFlow not installed.")
    except Exception as e:
        print(f"\n[2] TensorFlow warning: {e}")

# ============================================================
# GARBAGE COLLECTION
# ============================================================

def force_garbage_collection(cycles=3):
    print("\n[4] Running garbage collection...")
    total = sum(gc.collect() for _ in range(cycles))
    print(f"    Garbage-collected objects: {total}")

# ============================================================
# DELETE USER / MODEL / DATA VARIABLES
# ============================================================

def delete_model_variables():
    print("\n[5] Deleting user/model/data variables...")

    try:
        from IPython import get_ipython
        ip = get_ipython()

        if ip is None:
            print("    Not running inside Jupyter.")
            return 0, []

        protected = {
            "os", "gc", "time", "psutil",
            "get_user_variables", "get_ram_usage", "print_variable_information",
            "clear_matplotlib_cache", "clear_tensorflow_memory", "force_garbage_collection",
            "delete_model_variables", "clear_cache_and_memory",
            "In", "Out", "get_ipython", "exit", "quit"
        }

        names = [name for name in list(ip.user_ns.keys())
                 if not name.startswith("_") and name not in protected]

        print(f"    Variables selected for deletion: {len(names)}")
        if names: print("    Deleting: " + ", ".join(sorted(names)))

        deleted = 0

        for name in names:
            try:
                del ip.user_ns[name]
                deleted += 1
            except Exception as e:
                print(f"    Could not delete {name}: {e}")

        print(f"    Successfully deleted: {deleted}")
        return deleted, names

    except Exception as e:
        print(f"    Variable cleanup warning: {e}")
        return 0, []

# ============================================================
# COMPLETE CLEANUP
# ============================================================

def clear_cache_and_memory():
    print("\n" + "=" * 70)
    print("COMPLETE MEMORY / CACHE / VARIABLE CLEANUP")
    print("=" * 70)

    # BEFORE
    before_ram = get_ram_usage()
    before_variables = get_user_variables()
    before_count = len(before_variables)

    print("\nBEFORE CLEANING")
    print("-" * 70)
    print(f"RAM usage       : {before_ram:.2f} MB")
    print(f"Variable count  : {before_count}")
    if before_variables: print("Variables       : " + ", ".join(sorted(before_variables.keys())))

    # CLEAN
    clear_matplotlib_cache()
    clear_tensorflow_memory()
    force_garbage_collection()
    deleted_count, deleted_names = delete_model_variables()

    # FINAL GARBAGE COLLECTION
    print("\n[6] Final garbage collection...")
    total = sum(gc.collect() for _ in range(5))
    print(f"    Garbage-collected objects: {total}")
    time.sleep(1)

    # AFTER
    after_ram = get_ram_usage()
    after_variables = get_user_variables()
    after_count = len(after_variables)

    print("\nAFTER CLEANING")
    print("-" * 70)
    print(f"RAM usage       : {after_ram:.2f} MB")
    print(f"Variable count  : {after_count}")
    if after_variables: print("Remaining       : " + ", ".join(sorted(after_variables.keys())))

    # SUMMARY
    ram_difference = before_ram - after_ram

    print("\n" + "=" * 70)
    print("CLEANUP SUMMARY")
    print("=" * 70)
    print(f"Variables before : {before_count}")
    print(f"Variables after  : {after_count}")
    print(f"Variables deleted: {deleted_count}")
    print("-" * 70)
    print(f"RAM before       : {before_ram:.2f} MB")
    print(f"RAM after        : {after_ram:.2f} MB")

    if ram_difference >= 0: print(f"RAM released     : {ram_difference:.2f} MB")
    else: print(f"RAM difference   : +{abs(ram_difference):.2f} MB")

    print("=" * 70)
    print("CLEANUP COMPLETED")
    print("=" * 70)

# ============================================================
# RUN
# ============================================================

clear_cache_and_memory()

In [ ]:
import os
import tensorflow as tf
from tensorflow.keras import layers, models, Input, Model, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical
import glob
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import scipy.sparse as sp
import matplotlib.pyplot as plt
from spektral.layers import GraphSageConv
from spektral.utils import normalized_adjacency
from sklearn.neighbors import kneighbors_graph
from sklearn.preprocessing import label_binarize
from tensorflow.keras.applications import VGG16
from sklearn.utils.class_weight import compute_class_weight

# --- Step 1: Normalize Adjacency Matrix ---
def normalize_adjacency(A):
    """
    Normalize the adjacency matrix for GCN: D^(-1/2) * A * D^(-1/2)
    where D is the degree matrix.
    """
    A = sp.csr_matrix(A)  # Ensure sparse matrix format
    D = np.array(A.sum(axis=1))  # Calculate degree matrix
    D_inv_sqrt = np.power(D, -0.5).flatten()
    D_inv_sqrt[np.isinf(D_inv_sqrt)] = 0.  # Handle division by zero
    D_inv_sqrt = sp.diags(D_inv_sqrt)  # Create diagonal matrix
    A_norm = D_inv_sqrt.dot(A).dot(D_inv_sqrt)  # Normalize adjacency matrix
    return A_norm

# --- Step 2: Image Data Preprocessing ---
data_dir = r"D:/Customised_CNN/dataset/Brain_Tumor/four_class"
IMG_SIZE_RAW = (112, 112)
IMG_SIZE_CNN = (150, 150)  # For CNN
SEED = 123

# Function to load and preprocess raw images
def load_raw_image(path, size=IMG_SIZE_RAW):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.convert_image_dtype(img, tf.float32)
    img = tf.image.resize(img, size, method=tf.image.ResizeMethod.BILINEAR)
    return tf.reshape(img, [-1])  # Flatten image to 1D vector

# --- Step 3: Prepare Data ---
class_names = sorted([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))])
class_to_idx = {c: i for i, c in enumerate(class_names)}

paths, labels_int = [], []
for c in class_names:
    cdir = os.path.join(data_dir, c)
    for p in glob.glob(os.path.join(cdir, "*")):
        if p.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")):
            paths.append(p); labels_int.append(class_to_idx[c])

paths = np.array(paths)
labels_int = np.array(labels_int, dtype=np.int32)
num_classes = len(class_names)

# Split into train, validation, and test
# Split into 70% train, 20% validation, and 10% unseen test
all_idx = np.arange(len(paths))

# First reserve 10% as completely unseen test data
idx_tr_va, idx_te = train_test_split(
    all_idx,
    test_size=0.10,
    random_state=SEED,
    stratify=labels_int
)

# Remaining data = 90%
# Validation must be 20% of original dataset
# 20 / 90 = 0.222222...
idx_tr, idx_va = train_test_split(
    idx_tr_va,
    test_size=2/9,
    random_state=SEED,
    stratify=labels_int[idx_tr_va]
)

print("\nDataset Split")
print("--------------------------------")
print(f"Total      : {len(all_idx)}")
print(f"Training   : {len(idx_tr)} ({len(idx_tr)/len(all_idx)*100:.2f}%)")
print(f"Validation : {len(idx_va)} ({len(idx_va)/len(all_idx)*100:.2f}%)")
print(f"Test       : {len(idx_te)} ({len(idx_te)/len(all_idx)*100:.2f}%)")

# ============================================================
# HANDLING IMBALANCED DATA AND RARE CASES
# ============================================================

# Class distribution for training, validation, and unseen test sets
def show_class_distribution(name, idx):
    print(f"\n{name} Class Distribution")
    print("-" * 50)
    for i, c in enumerate(class_names):
        n = np.sum(labels_int[idx] == i)
        print(f"{c:<20}: {n:4d} ({n/len(idx)*100:.2f}%)")

show_class_distribution("TRAINING", idx_tr)
show_class_distribution("VALIDATION", idx_va)
show_class_distribution("UNSEEN TEST", idx_te)

# Compute balanced class weights using training data only
train_labels = labels_int[idx_tr]

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_labels),
    y=train_labels
)

class_weights = {int(c): float(w) for c, w in zip(np.unique(train_labels), weights)}

print("\nClass Weights")
print("-" * 50)
for i, c in enumerate(class_names):
    print(f"{c:<20}: {class_weights[i]:.4f}")

# Convert labels to one-hot
y_onehot = to_categorical(labels_int, num_classes=num_classes).astype(np.float32)

# --- Step 4: Data Augmentation ---
datagen = ImageDataGenerator(
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Fit on training data
#datagen.fit(paths)

# --- Step 5: CNN Backbone (Feature Extraction) ---
def build_cnn_backbone(input_shape=(IMG_SIZE_CNN[0], IMG_SIZE_CNN[1], 3), feat_dim=512):
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation="relu", input_shape=input_shape),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Conv2D(64, (3, 3), activation="relu"),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Conv2D(128, (3, 3), activation="relu"),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Flatten(),
        layers.Dense(feat_dim, activation="relu", name="feat")
    ])
    return model

cnn_backbone = build_cnn_backbone()

# --- Step 6: Feature Extraction ---
def extract_cnn_features(paths_np):
    feats = []
    for path in paths_np:
        img = tf.io.read_file(path)
        img = tf.image.decode_image(img, channels=3, expand_animations=False)
        img = tf.image.convert_image_dtype(img, tf.float32)
        img = tf.image.resize(img, IMG_SIZE_CNN)
        img = tf.reshape(img, (1, IMG_SIZE_CNN[0], IMG_SIZE_CNN[1], 3))
        feat = cnn_backbone(img).numpy()
        feats.append(feat)
    return np.vstack(feats)

X_cnn = extract_cnn_features(paths)

# --- Step 7: Construct Graph using k-NN ---
# Construct k-NN graph as adjacency matrix
knn_graph = kneighbors_graph(X_cnn, n_neighbors=10, mode='connectivity', include_self=True)
A_mut = knn_graph.toarray()  # Convert to dense matrix
A_norm = normalize_adjacency(A_mut)  # Normalize adjacency matrix

# --- Step 8: Hyperparameter Tuning ---
from tensorflow.keras.optimizers import Adam
model = build_cnn_backbone()
model.compile(optimizer=Adam(learning_rate=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])

# --- Step 9: Build the GCN Model ---
def build_gcn(input_shape, num_classes, N):
    X_in = Input(shape=(input_shape,), name="X_in")
    A_in = Input((N,), sparse=True, name="A_in")
    h1 = GraphSageConv(32, activation="relu", kernel_regularizer=regularizers.l2(5e-4))([X_in, A_in])
    h1 = layers.Dropout(0.3)(h1)
    res = layers.Dense(32, use_bias=False)(X_in)
    h1 = layers.Add()([h1, res])

    out = GraphSageConv(num_classes, activation="softmax")([h1, A_in])

    model = Model(inputs=[X_in, A_in], outputs=out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(5e-3),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

input_shape = X_cnn.shape[1]  # CNN feature size
gcn_model = build_gcn(input_shape=input_shape, num_classes=num_classes, N=X_cnn.shape[0])
gcn_model.summary()

# --- Step 10: Regularization ---
callbacks = [
    EarlyStopping(monitor="val_loss", patience=20, restore_best_weights=True),
    ModelCheckpoint("best_gcn_ensemble.keras", monitor="val_loss"),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5)
]

# --- Step 11: Train the Model ---
X_train = X_cnn[idx_tr]
X_val = X_cnn[idx_va]
X_test = X_cnn[idx_te]

A_train = A_norm[idx_tr]
A_val = A_norm[idx_va]
A_test = A_norm[idx_te]

y_train = to_categorical(
    labels_int[idx_tr],
    num_classes=num_classes
)

y_val = to_categorical(
    labels_int[idx_va],
    num_classes=num_classes
)

y_test = to_categorical(
    labels_int[idx_te],
    num_classes=num_classes
)

mask_bool_train = np.ones(len(y_train))
mask_bool_val = np.ones(len(y_val))
mask_bool_test = np.ones(len(y_test))

# Train the model
gcn_model.fit(
    x=[X_train, A_train],  # Train on training data
    y=y_train,             # Train on training labels
    batch_size=8,          # Adjust the batch size
    epochs=100,            # Increase epochs for better training
    validation_data=([X_val, A_val], y_val),  # Pass validation data explicitly
    class_weight=class_weights,  # Handle class imbalance
    verbose=1,
    callbacks=callbacks
)

# --- Step 12: Evaluation ---
def evaluate_mask(mask_name, mask_bool, X_data, A_data, y_data):
    # Ensure the sample_weight is correctly applied
    print(f"Evaluating {mask_name} with sample weight shape: {mask_bool.shape}")
    
    # Use the correct subset of labels for evaluation
    loss, acc = gcn_model.evaluate([X_data, A_data], 
                                   to_categorical(y_data, num_classes=num_classes),
                                   sample_weight=mask_bool.astype(np.float32),
                                   batch_size=32, verbose=0)
    print(f"[{mask_name}] loss={loss:.4f}  acc={acc:.4f}")
    
    y_prob = gcn_model.predict([X_data, A_data], batch_size=32, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)
    y_true = y_data
    m = mask_bool.astype(bool)  # Convert mask_bool to boolean for indexing

    # Print classification report and confusion matrix
    print(f"\n[{mask_name}] Per-Class Classification Report:")
    print(classification_report(
        y_true[m],
        y_pred[m],
        target_names=class_names,
        digits=4,
        zero_division=0
    ))

    cm = confusion_matrix(y_true[m], y_pred[m])
    print(f"[{mask_name}] Confusion Matrix:\n{cm}")

    # Class-wise false positives, false negatives, Type I and Type II errors
    print(f"\n[{mask_name}] Class-wise Error Analysis")
    print("-" * 80)
    print(f"{'Class':<20}{'FP':<8}{'FN':<8}{'Type I (%)':<15}{'Type II (%)':<15}")

    for i, c in enumerate(class_names):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP
        FP = np.sum(cm[:, i]) - TP
        TN = np.sum(cm) - TP - FP - FN

        type_I = FP / (FP + TN) if (FP + TN) > 0 else 0
        type_II = FN / (FN + TP) if (FN + TP) > 0 else 0

        print(f"{c:<20}{FP:<8}{FN:<8}{type_I*100:<15.2f}{type_II*100:<15.2f}")
    
    # Confusion matrix heatmap
    plt.figure(figsize=(6, 6))
    plt.imshow(cm, cmap="Blues")
    plt.title(f"GCN Confusion Matrix ({mask_name})")
    plt.colorbar()
    plt.xticks(range(num_classes), class_names, rotation=45, ha="right")
    plt.yticks(range(num_classes), class_names)
    plt.xlabel("Predicted"); plt.ylabel("True")
    for i in range(num_classes):
        for j in range(num_classes):
            plt.text(j, i, cm[i, j], ha="center", va="center", color="red")
    plt.tight_layout()
    plt.show()

    # Optional ROC
    try:
        y_bin  = label_binarize(y_true[m], classes=np.arange(num_classes))
        y_pred_bin = y_prob[m]
        fpr, tpr, roc_auc = {}, {}, {}
        for i in range(num_classes):
            fpr[i], tpr[i], _ = roc_curve(y_bin[:, i], y_pred_bin[:, i])
            roc_auc[i] = auc(fpr[i], tpr[i])
        fpr["micro"], tpr["micro"], _ = roc_curve(y_bin.ravel(), y_pred_bin.ravel())
        roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

        plt.figure(figsize=(8, 6))
        for i in range(num_classes):
            plt.plot(fpr[i], tpr[i], label=f"{class_names[i]} (AUC={roc_auc[i]:.2f})")
        plt.plot(fpr["micro"], tpr["micro"], linestyle="--", label=f"Micro (AUC={roc_auc['micro']:.2f})")
        plt.plot([0,1],[0,1],"k--")
        plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
        plt.title(f"GCN ROC Curves ({mask_name}, Raw)")
        plt.legend(loc="lower right")
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"[ROC] Skipped ({e})")

# Evaluate on validation and test sets
# ============================================================
# CHANGE 3: EVALUATE ACTUAL TEST SET
# ============================================================

print("[STEP 6] Evaluation:")

evaluate_mask(
    "VALIDATION",
    mask_bool_val,
    X_val,
    A_val,
    labels_int[idx_va]
)

evaluate_mask(
    "TEST",
    mask_bool_test,
    X_test,
    A_test,
    labels_int[idx_te]
)

# --- Step 13: Graph Metrics Calculation ---
def graph_metrics(A, X_cnn):
    """
    Calculate and return various metrics of the graph
    """
    N = A.shape[0]  # Number of nodes
    
    # Feature dimension (F) is the number of features per node
    F = X_cnn.shape[1]
    
    # Remove self-loops from the adjacency matrix (set diagonal to 0)
    np.fill_diagonal(A, 0)
    
    # Calculate the number of edges (undirected, no self-loops)
    edges = np.sum(A > 0) / 2  # Each edge is counted twice in an undirected graph
    
    # Average degree (no self-loops)
    degree = np.sum(A > 0, axis=1)  # Sum the edges for each node
    avg_degree = np.mean(degree)  # Average degree
    
    # Graph density: density = (2 * number of edges) / (N * (N - 1))
    max_edges = N * (N - 1) / 2
    graph_density = edges / max_edges if max_edges > 0 else 0
    
    # Find connected components
    from scipy.sparse.csgraph import connected_components
    n_components, labels = connected_components(A, directed=False, return_labels=True)
    
    # Size of the largest connected component
    component_sizes = [np.sum(labels == i) for i in range(n_components)]
    largest_cc_size = max(component_sizes) if component_sizes else 0
    
    # Return the results as a dictionary
    return {
        "Nodes (N)": N,
        "Feature Dimension (F)": F,
        "Undirected Edges (no self-loops)": int(edges),
        "Avg Degree (no self-loops)": avg_degree,
        "Graph Density": graph_density,
        "Connected Components": n_components,
        "Largest CC Size": largest_cc_size
    }

# Calculate and display the graph metrics
metrics = graph_metrics(A_mut, X_cnn)

# Display the metrics as a table
import pandas as pd

metrics_df = pd.DataFrame([metrics])
print(metrics_df)